In [1]:
import os
import cv2
import numpy as np

IMG_SIZE = (64, 64)

def load_images_from_folder(folder, label):
    images = []
    labels = []
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        img = cv2.imread(path)
        if img is not None:
            img = cv2.resize(img, IMG_SIZE)
            images.append(img)
            labels.append(label)
    return images, labels

male_imgs, male_labels = load_images_from_folder('dataset/Training/male', 0)
female_imgs, female_labels = load_images_from_folder('dataset/Training/female', 1)

X = np.array(male_imgs + female_imgs) / 255.0  # Normalize
y = np.array(male_labels + female_labels)


In [13]:
print(X)


[[[[0.08627451 0.15294118 0.18823529]
   [0.15294118 0.21960784 0.27058824]
   [0.22745098 0.29803922 0.36470588]
   ...
   [0.32941176 0.41960784 0.50588235]
   [0.30196078 0.39607843 0.47058824]
   [0.25882353 0.35294118 0.41960784]]

  [[0.07843137 0.13333333 0.18039216]
   [0.1254902  0.18823529 0.24313725]
   [0.20784314 0.2745098  0.34509804]
   ...
   [0.31372549 0.39607843 0.49411765]
   [0.28627451 0.38039216 0.45882353]
   [0.24705882 0.34117647 0.41176471]]

  [[0.05882353 0.11372549 0.16078431]
   [0.11764706 0.17647059 0.23529412]
   [0.20784314 0.26666667 0.34509804]
   ...
   [0.2745098  0.35294118 0.46666667]
   [0.24705882 0.33333333 0.43137255]
   [0.21568627 0.30196078 0.38823529]]

  ...

  [[0.63921569 0.61960784 0.3254902 ]
   [0.62745098 0.61176471 0.31764706]
   [0.61568627 0.60392157 0.31372549]
   ...
   [0.33333333 0.4745098  0.69019608]
   [0.34117647 0.48235294 0.69411765]
   [0.35294118 0.49411765 0.70196078]]

  [[0.63529412 0.61176471 0.3254902 ]
   [0.6

In [15]:
print(y)

[0 0 0 ... 1 1 1]


In [17]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y = to_categorical(y, num_classes=2)  # One-hot encode

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [21]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64, 64, 3)),   
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')  # 2 classes
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       802,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 822,402 (3.14 MB)

 Trainable params: 822,402 (3.14 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
print(model)

<Sequential name=sequential_1, built=True>


In [25]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1)


Epoch 1/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 57s 130ms/step - accuracy: 0.7984 - loss: 0.4397 - val_accuracy: 0.9272 - val_loss: 0.2119
Epoch 2/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 115s 218ms/step - accuracy: 0.9263 - loss: 0.1922 - val_accuracy: 0.8963 - val_loss: 0.2570
Epoch 3/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 128s 180ms/step - accuracy: 0.9355 - loss: 0.1654 - val_accuracy: 0.9346 - val_loss: 0.1932
Epoch 4/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 57s 150ms/step - accuracy: 0.9520 - loss: 0.1309 - val_accuracy: 0.9243 - val_loss: 0.2140
Epoch 5/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 45s 118ms/step - accuracy: 0.9629 - loss: 0.1078 - val_accuracy: 0.9493 - val_loss: 0.1639
Epoch 6/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 47s 122ms/step - accuracy: 0.9677 - loss: 0.0934 - val_accuracy: 0.9419 - val_loss: 0.1601
Epoch 7/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 89s 141ms/step - accuracy: 0.9729 - loss: 0.0750 - val_accuracy: 0.9485 - val_loss: 0.1643
Epoch 8/10
383/383 ━━━━━━━━━━━━━━━━━━━━ 85s 148ms/step - accuracy: 0.9811 - loss:

In [134]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.2f}")




107/107 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.9162 - loss: 0.2318
Test Accuracy: 0.92
